### Scoring Algorithm

In [ ]:
import pandas as pd
import openpyxl
import os

# Load Shiftly Template
shiftly_wb = openpyxl.load_workbook("Shiftly Template.xlsm", keep_vba=True)
req_entry_sheet = shiftly_wb['Requirements Entry']

# Import applicant data
df = pd.read_excel('applicants.csv')

# Generate question template sheet names
num_questions = 6
question_sheets = [f'Question_{i}' for i in range(1, num_questions + 1)]
num_questions = len(question_sheets)

# ── Load question configs from each sheet ─────────────────────────────────────
question_configs = {}  # { question_text: {option: score, ...} }

for sheet_name in question_sheets:
    ws = shiftly_wb[sheet_name]

    question_text = ws['A2'].value

    option_score_map = {}
    row = 5
    while ws[f'A{row}'].value is not None and str(ws[f'A{row}'].value).strip() != '':
        option = str(ws[f'A{row}'].value).strip()
        score  = ws[f'B{row}'].value
        option_score_map[option] = score
        row += 1

    question_configs[question_text] = option_score_map

print(question_configs)

# Score cutoff
score_cutoff = req_entry_sheet['A7'].value

# Clean column names
df.columns = df.columns.str.strip()
df_score = df.copy()

# ── Identify weekly availability columns ──────────────────────────────────────
availability_cols = [c for c in df.columns if 'weekly availability' in c.lower()]
print(f"Found availability columns: {availability_cols}")

# ── Fully dynamic scoring (matches by column header name) ─────────────────────
dynamic_score_cols = []

for question_text, option_score_map in question_configs.items():
    score_col = f'{question_text} (score)'

    # Directly match by column name instead of value subset detection
    if question_text in df_score.columns:
        # Score = sum of scores for each selected option (handles multi-select)
        df_score[score_col] = df_score[question_text].apply(
            lambda cell: sum(
                option_score_map.get(val.strip(), 0)
                for val in str(cell).split(',')
            ) if pd.notna(cell) else 0
        )
        dynamic_score_cols.append(score_col)
    else:
        print(f"WARNING: No matching column found in CSV for question: '{question_text}'")

# Total score is purely the sum of all dynamic question scores
df_score['Score'] = df_score[dynamic_score_cols].sum(axis=1)

# Sort by score and assign rank
df_score = df_score.sort_values(by='Score', ascending=False).reset_index(drop=True)
df_score.insert(0, 'Rank', df_score.index + 1)

# ── Build final output column order ───────────────────────────────────────────
fixed_cols  = ['Rank', 'Full Name', 'Email Address', 'Grade']
answer_cols = list(question_configs.keys())
score_cols  = [f'{q} (score)' for q in question_configs.keys()]
tail_cols   = ['Score'] + availability_cols  # availability appended after Score

final_cols = fixed_cols + answer_cols + score_cols + tail_cols
final_cols = [c for c in final_cols if c in df_score.columns]

df_score_top = df_score[df_score["Score"] >= score_cutoff][final_cols]

# ── Save to Excel ──────────────────────────────────────────────────────────────
output_wb = openpyxl.Workbook()
output_ws = output_wb.active
output_ws.title = "Ranked Applicants"

for col_idx, col_name in enumerate(df_score_top.columns, start=1):
    output_ws.cell(row=1, column=col_idx, value=col_name)

for row_idx, row in enumerate(df_score_top.itertuples(index=False), start=2):
    for col_idx, value in enumerate(row, start=1):
        output_ws.cell(row=row_idx, column=col_idx, value=value)

output_wb.save("ranked_applicants.xlsx")
print("Saved ranked_applicants.xlsx")

{'How much experience have you had working with youth? (e.g., tutoring, mentoring, childcare, coaching, etc.)': {'0 hours': 0, '1-9 hours': 1, '10-20 hours': 2, '20+ hours': 3}, 'How much leadership experience do you have?  (volunteering, clubs, sports, work experience, etc.)': {'0 hours': 0, '1-9 hours': 1, '10-20 hours': 2, '20+ hours': 3}, 'Which of the following subjects have you taken or are currently taking? (Check all that apply)': {'Programming': 1, 'Physics': 1, 'Biology': 1, 'Chemistry': 1, 'Science': 1, 'Math': 1, 'Calculus': 1, 'Earth & Space Science': 1, 'Robotics': 1}, 'What technical skills do you have? (Check all that apply)': {'Coding': 1, 'Robotics': 1, 'Graphic Design': 1, 'Video Editing': 1, '3D Modeling': 1}, 'Have you previously been a volunteer with UWaterloo Engineering Outreach Summer Camps?': {'Yes': 2, 'No': 0}, 'Have you previously been a camper at UWaterloo Engineering Outreach Summer Camps?': {'Yes': 1, 'No': 0}}
Found availability columns: ['Please enter 